## importing deployment libraries

In [1]:
## importing deployment libraries
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

## locating saved model

In [2]:
## locating saved model
FILE_DIR = Path.cwd() / "Data_Collection"
MODEL_PATH = FILE_DIR / "agriyield_model_pipeline.joblib"
METADATA_PATH = FILE_DIR / "model_metadata.json"
DATA_PATH = FILE_DIR / "eligible_model_data.csv"

for path in (MODEL_PATH, METADATA_PATH, DATA_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. "
            "Run 04_Final_Model_Updated.ipynb first and check FILE_DIR."
        )

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

model = joblib.load(MODEL_PATH)
history_df = pd.read_csv(DATA_PATH)

model_name = metadata.get("model_name", metadata.get("validation_winner", "Selected model"))
target_col = metadata.get("target", "Yield_kg_ha")
numeric_features = metadata.get("numeric_features", [])
categorical_features = metadata.get("categorical_features", [])
model_input_features = metadata.get("model_input_features", numeric_features + categorical_features)

print("Loaded model:", model_name)
print("Prediction target:", target_col)
print("Model inputs:", model_input_features)
print("Historical rows:", len(history_df))

Loaded model: Ridge Regression (lags only)
Prediction target: Yield_kg_ha
Model inputs: ['Yield_lag_1', 'Yield_lag_2', 'Yield_lag_3', 'Yield_rolling_mean_3']
Historical rows: 23189


## validating required historical columns

In [3]:
## validating required historical columns
required_history_cols = {
    "Area", "Item", "Year", target_col,
    "Yield_lag_1", "Yield_lag_2", "Yield_lag_3", "Yield_rolling_mean_3"
}
missing_history_cols = required_history_cols - set(history_df.columns)
if missing_history_cols:
    raise ValueError(f"Historical data is missing required columns: {sorted(missing_history_cols)}")

history_df["Year"] = pd.to_numeric(history_df["Year"], errors="raise").astype(int)
missing_feature_cols = set(model_input_features) - set(history_df.columns)
if missing_feature_cols:
    raise ValueError(f"Model input columns are missing: {sorted(missing_feature_cols)}")

print("Historical data and model feature columns are valid.")

Historical data and model feature columns are valid.


## creating a one-year-ahead prediction function

In [4]:
## creating a one-year-ahead prediction function
def get_next_year_features(area, crop):
    """Build one row of model features from the latest three consecutive yields."""
    subset = history_df[
        (history_df["Area"].astype(str) == str(area)) &
        (history_df["Item"].astype(str) == str(crop))
    ].copy()

    if subset.empty:
        raise ValueError(f"No historical records found for area={area!r}, crop={crop!r}.")

    subset = subset.sort_values("Year")
    latest_year = int(subset["Year"].max())
    expected_years = [latest_year - 2, latest_year - 1, latest_year]
    recent = subset[subset["Year"].isin(expected_years)].drop_duplicates("Year")

    if sorted(recent["Year"].tolist()) != expected_years:
        raise ValueError(
            f"Three consecutive annual yields are not available for {area} / {crop} "
            f"through {latest_year}."
        )

    yields_by_year = recent.set_index("Year")[target_col]
    lag_1 = float(yields_by_year.loc[latest_year])
    lag_2 = float(yields_by_year.loc[latest_year - 1])
    lag_3 = float(yields_by_year.loc[latest_year - 2])

    feature_row = {
        "Yield_lag_1": lag_1,
        "Yield_lag_2": lag_2,
        "Yield_lag_3": lag_3,
        "Yield_rolling_mean_3": float(np.mean([lag_1, lag_2, lag_3])),
        "Area": str(area),
        "Item": str(crop)
    }
    feature_df = pd.DataFrame([feature_row])

    missing_inputs = set(model_input_features) - set(feature_df.columns)
    if missing_inputs:
        raise ValueError(
            f"Cannot build required model inputs: {sorted(missing_inputs)}. "
            "Check the feature metadata saved by the final-model notebook."
        )

    return latest_year + 1, feature_df[model_input_features], expected_years

## listing available areas and crops

In [5]:
## listing available areas and crops
available_areas = sorted(history_df["Area"].dropna().astype(str).unique().tolist())
available_crops = sorted(history_df["Item"].dropna().astype(str).unique().tolist())

print("Available areas:", len(available_areas))
print(available_areas[:20])
print("\nAvailable crops:")
print(available_crops)

Available areas: 196
['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Australia and New Zealand', 'Austria', 'Azerbaijan', 'Bahamas', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belgium-Luxembourg', 'Belize', 'Benin', 'Bhutan']

Available crops:
['Maize (corn)', 'Rice', 'Wheat']


## generating a sample prediction

In [6]:
## generating a sample prediction
# Change these values to an area and crop present in the historical dataset.
SAMPLE_AREA = available_areas[0]
SAMPLE_CROP = available_crops[0]

forecast_year, sample_X, source_years = get_next_year_features(SAMPLE_AREA, SAMPLE_CROP)
sample_prediction = float(model.predict(sample_X)[0])

print("Area:", SAMPLE_AREA)
print("Crop:", SAMPLE_CROP)
print("Historical years used:", source_years)
print("Forecast year:", forecast_year)
print(f"Predicted yield: {sample_prediction:,.2f} kg/ha")

Area: Afghanistan
Crop: Maize (corn)
Historical years used: [2021, 2022, 2023]
Forecast year: 2024
Predicted yield: 2,611.96 kg/ha


In [7]:
# ## creating a reusable prediction result
def predict_crop_yield(area, crop):
    """Return a structured prediction result for one area and crop."""
    forecast_year, input_features, source_years = get_next_year_features(area, crop)
    prediction = float(model.predict(input_features)[0])

    if not np.isfinite(prediction):
        raise ValueError("The model returned a non-finite prediction.")

    return {
        "Area": str(area),
        "Crop": str(crop),
        "Forecast_Year": int(forecast_year),
        "Predicted_Yield_kg_ha": round(prediction, 2),
        "Historical_Years_Used": source_years,
        "Model": model_name,
        "Interpretation": (
            "One-year-ahead estimate based on previous observed yields; "
            "not a guarantee and not weather-aware."
        )
    }

prediction_result = predict_crop_yield(SAMPLE_AREA, SAMPLE_CROP)
display(pd.DataFrame([prediction_result]))

,Area,Crop,Forecast_Year,Predicted_Yield_kg_ha,Historical_Years_Used,Model,Interpretation
0,Afghanistan,Maize (corn),2024,2611.96,"[2021, 2022, 2023]",Ridge Regression (lags only),One-year-ahead estimate based on previous obse...


## interactive crop yield prediction

In [8]:
## interactive crop yield prediction

# Enter the area and crop
area_input = input("Enter country/area (e.g., Oman): ").strip()
crop_input = input("Enter crop (e.g., Maize (corn), Rice, Wheat): ").strip()

# Generate prediction
try:
    result = predict_crop_yield(area_input, crop_input)

    print("\n========== AGRIYIELD AI ==========")
    print(f"Area             : {result['Area']}")
    print(f"Crop             : {result['Crop']}")
    print(f"Forecast year    : {result['Forecast_Year']}")
    print(f"Predicted yield  : {result['Predicted_Yield_kg_ha']:,.2f} kg/ha")
    print(f"Historical years : {result['Historical_Years_Used']}")
    print(f"Model            : {result['Model']}")
    print("=================================")

except ValueError as error:
    print(f"\nUnable to generate prediction: {error}")


Enter country/area (e.g., Oman):  Bangladesh
Enter crop (e.g., Maize (corn), Rice, Wheat):  Wheat



========== AGRIYIELD AI ==========
Area             : Bangladesh
Crop             : Wheat
Forecast year    : 2024
Predicted yield  : 3,517.41 kg/ha
Historical years : [2021, 2022, 2023]
Model            : Ridge Regression (lags only)


## testing the deployment prediction function

In [9]:
## testing the deployment prediction function
test_result = predict_crop_yield(SAMPLE_AREA, SAMPLE_CROP)

assert test_result["Area"] == SAMPLE_AREA
assert test_result["Crop"] == SAMPLE_CROP
assert isinstance(test_result["Forecast_Year"], int)
assert np.isfinite(test_result["Predicted_Yield_kg_ha"])

print("Deployment smoke test passed.")

Deployment smoke test passed.


## saving a sample deployment prediction

In [10]:
## saving a sample deployment prediction
prediction_output_path = FILE_DIR / "sample_deployment_prediction.csv"
pd.DataFrame([prediction_result]).to_csv(prediction_output_path, index=False)
print("Saved sample prediction:", prediction_output_path)

Saved sample prediction: C:\Users\acer\capstone_Project\Agriculture_Yield_Prediction\Crop_Field_Prediction_Using_ML\Data_Collection\sample_deployment_prediction.csv
